# 两层 GCN —— 标准库实现

---
## 目录

| 节 | 内容 |
|---|---|
| 1 | 矩阵（list of list） |
| 2 | 数据 |
| 3 | 初始化 |
| 4 | 前向 / 反向 |
| 5 | 评估 |
| 6 | Adam |
| 7 | 训练 |
| 8 | 主流程 |
| — | 运行 |
| — | 结果分析 |

---

## 源码说明
### 公式（Kipf & Welling 2016）
预处理：

```text
A_tilde = A + I
D_tilde = diag(A_tilde 的行和)
A_hat   = D_tilde^(-1/2) · A_tilde · D_tilde^(-1/2)
```
前向：

```text
Z = softmax( A_hat · ReLU( A_hat · X · W_0 ) · W_1 )
```
损失：

```text
L_0 = -Σ_{i∈train} Σ_f  Y[i,f] · ln Z[i,f]
L   = L_0 + lambda_reg · (||W_0||² + ||W_1||²) / 2
```
### 变量名
```text
X        (N, C)  节点特征，C 是输入通道数
A        (N, N)  邻接矩阵
A_hat    (N, N)  归一化邻接矩阵
W_0      (C, H)  输入到隐藏层的权重
W_1      (H, F)  隐藏层到输出的权重
H_1      (N, H)  隐藏层激活
Z        (N, F)  行方向 softmax 后的类别概率
Y        (N, F)  one-hot 标签
N, C, H, F       节点数、输入维度、隐藏维度、类别数
```
超参数：Adam，lr=0.01，最多 200 轮，
dropout 0.5，L2 = 5e-4，隐藏层 16 维，Glorot 初始化，特征按行归一化。


## 0 模块导入

只 import 标准库。

In [1]:
import argparse
import csv
import json
import math
import os
import random
import time
from math import sumprod as dot
from pathlib import Path

CWD = Path.cwd()
HERE = CWD.parent if CWD.name == 'notebooks' else CWD


## 1 矩阵（list of list）

In [2]:
def zeros(rows, cols):
    """生成 rows×cols 的全零稠密矩阵。"""
    return [[0.0] * cols for _ in range(rows)]

def transpose(A):
    """矩阵转置。返回的是 tuple 的行，不能再原地修改。"""
    return list(zip(*A))

def matmul(A, B):
    """A(m×n) · B(n×p) —— 把 B 的列摊开，逐行点积。"""
    cols_of_B = list(zip(*B))            # `zip(*B)` 直接按列取，省掉一次真正的转置
    return [[dot(row, col) for col in cols_of_B] for row in A]


def matmul_t(A, B):
    """Aᵀ · B，A 是 (n×m)、B 是 (n×p)，结果 (m×p)。
    用于反向求导
    """
    cols_of_B = list(zip(*B))
    return [[dot(col_a, col_b) for col_b in cols_of_B] for col_a in zip(*A)]

## 2 数据

In [3]:
def load_dataset(name):
    """从 <name>/ 读取稠密数据，返回 X, A, Y(label) 与 mask(train,val,test)。"""
    data_dir = os.path.join(HERE, name)
    with open(os.path.join(data_dir, "meta.json"), encoding="utf-8") as f:
        num_classes = json.load(f)["num_classes"]

    # ---- 节点：标签 + 划分 ----
    labels, splits = [], []
    with open(os.path.join(data_dir, "nodes.csv"), newline="",
              encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)                                # 跳过表头
        for _node_id, split, label in reader:
            labels.append(int(label))
            splits.append(split)

    num_nodes = len(labels)

    # one-hot：Y[i, labels[i]] = 1；无标签节点（label = -1）整行为 0
    Y = zeros(num_nodes, num_classes)
    for i, label in enumerate(labels):
        if label >= 0:
            Y[i][label] = 1.0

    train_mask = [s == "train" for s in splits]
    val_mask = [s == "val" for s in splits]
    test_mask = [s == "test" for s in splits]

    # ---- 特征：稠密宽表，第一列是 node_id，按 0..N-1 顺序排列 ----
    X = []
    with open(os.path.join(data_dir, "features_dense.csv"), newline="",
              encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            X.append([float(v) for v in row[1:]])

    # ---- 邻接矩阵：稠密 N×N ----
    A = zeros(num_nodes, num_nodes)
    with open(os.path.join(data_dir, "edges.csv"), newline="",
              encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for u, v, _w in reader:
            A[int(u)][int(v)] = 1.0
            A[int(v)][int(u)] = 1.0

    # 数据里可能自带自环；先清零，后面统一加 I，避免自环被算成 2
    for i in range(num_nodes):
        A[i][i] = 0.0

    return X, A, Y, train_mask, val_mask, test_mask


def row_normalize(X):
    """特征按行归一化（论文 §5.1：「(row-)normalize input feature vectors」）。

    全零行保持全零，不做 0/0。
    """
    for row in X:
        total = sum(row)
        if total > 0.0:
            for j in range(len(row)):
                row[j] /= total
    return X


def build_A_hat(A):
    """A_hat = D_tilde^(-1/2) (A + I) D_tilde^(-1/2)，稠密实现只有三行。"""
    num_nodes = len(A)
    for i in range(num_nodes):
        A[i][i] += 1.0                              # Ã = A + I

    d_inv_sqrt = [1.0 / math.sqrt(sum(row)) for row in A]   # D̃^(-1/2)

    # d[i] 乘在左边、d[j] 乘在右边，等价于左乘 D̃^(-1/2)、右乘 D̃^(-1/2)
    for i in range(num_nodes):
        row = A[i]
        d_i = d_inv_sqrt[i]
        A[i] = [d_i * v * d_inv_sqrt[j] for j, v in enumerate(row)]
    return A

## 3 初始化

> Glorot 均匀初始化，界是 √(6/(fan_in+fan_out))。(这个有严格的数学证明)

In [4]:
def glorot(rows, cols, rng):
    """Glorot 均匀初始化：U(-sqrt(6/(fan_in+fan_out)), +...)。
    用来初始化神经网络权重矩阵，是 GCN 里给卷积层权重用的。
    目标：让每一层输入、输出的方差尽量保持稳定，避免梯度爆炸 / 消失。
    - `rows`：权重矩阵**输出维度（fan_out）**
    - `cols`：权重矩阵**输入维度（fan_in）**
    - `rng`：随机数生成器（比如 `np.random.RandomState`）
    - `bound = sqrt(6/(fan_in + fan_out))`：均匀分布上下界
    - 返回：`rows × cols` 的二维列表（权重矩阵），每个值从 \([-bound, bound]\) 均匀采样
    """
    bound = math.sqrt(6.0 / (rows + cols))
    return [[rng.uniform(-bound, bound) for _ in range(cols)]
            for _ in range(rows)]

<>:9: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<>:9: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
C:\Users\10740\AppData\Local\Temp\ipykernel_3884\182811213.py:9: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
  - 返回：`rows × cols` 的二维列表（权重矩阵），每个值从 \([-bound, bound]\) 均匀采样


## 4 前向 / 反向

> `AX` 是预算好的 `A_hat · X`：X 训练期不变，每轮重算要 O(N²C)。
> 代价是**输入侧无法做 dropout**（一加 AX 就失效），所以 dropout 只加在
> 隐藏层。反向里 ReLU 的导数由 `pre_1 > 0` 给出，不需要额外掩码。

In [5]:
def softmax(logits):
    """按行 softmax，先减去行最大值以免 exp 溢出。"""
    out = []
    for row in logits:
        top = max(row)
        exp_row = [math.exp(v - top) for v in row]
        total = sum(exp_row)
        out.append([v / total for v in exp_row])
    return out

def forward(AX, A_hat, W_0, W_1, keep_prob, rng, training):
    """返回 Z 以及反向需要的中间量。
    AX 是预先算好的 A_hat · X —— 因为 X 整个训练过程不变，
    """
    pre_1 = matmul(AX, W_0)                         # (N,C)(C,H) -> (N,H)
    H_1 = [[v if v > 0.0 else 0.0 for v in row] for row in pre_1]   # ReLU

    if training and keep_prob < 1.0:
        # 反向缩放（inverted dropout）：保留的单元除以 keep_prob，
        # 这样测试时不用再缩放
        mask = [[(1.0 / keep_prob) if rng.random() < keep_prob else 0.0
                 for _ in row] for row in H_1]
        H_1_in = [[v * m for v, m in zip(row, m_row)]
                  for row, m_row in zip(H_1, mask)]
    else:
        mask = None
        H_1_in = H_1

    A_H = matmul(A_hat, H_1_in)                     # Â · H_1_in，反向算 dW_1 还要用
    Z = softmax(matmul(A_H, W_1))                   # (N,H)(H,F) -> (N,F)
    cache = (AX, A_hat, W_0, W_1, pre_1, A_H, mask)
    return Z, cache


def backward(Z, Y, train_mask, cache, lambda_reg):
    """交叉熵 + L2 的梯度，返回 (d_W_0, d_W_1) 与 L_0。"""
    AX, A_hat, W_0, W_1, pre_1, A_H, mask = cache
    num_train = sum(train_mask)

    # 只在训练节点上算交叉熵，取平均而不是求和，
    # 这样学习率不随训练集大小变化。softmax + 交叉熵对 logits 的梯度
    # 就是 (Z - Y) / N_train，一起在这里算掉。
    L_0 = 0.0
    d_logits = zeros(len(Z), len(Z[0]))
    for i, (z_row, y_row) in enumerate(zip(Z, Y)):
        if not train_mask[i]:
            continue                                # 非训练节点梯度恒为 0
        for f in range(len(z_row)):
            L_0 -= y_row[f] * math.log(z_row[f] + 1e-12)
            d_logits[i][f] = (z_row[f] - y_row[f]) / num_train
    L_0 /= num_train

    # ---- W_1 的梯度 ----
    # logits = (A_hat · H_1_in) · W_1  =>  dW_1 = (A_hat · H_1_in)ᵀ · d_logits
    d_W_1 = matmul_t(A_H, d_logits)
    for i in range(len(d_W_1)):
        for j in range(len(d_W_1[i])):
            d_W_1[i][j] += lambda_reg * W_1[i][j]

    # ---- W_0 的梯度 ----
    # 反传到 H_1_in，再乘 dropout 掩码，再过 ReLU
    d_H_1_in = matmul(matmul_t(A_hat, d_logits), transpose(W_1))
    if mask is not None:
        d_H_1 = [[v * m for v, m in zip(row, m_row)]
                 for row, m_row in zip(d_H_1_in, mask)]
    else:
        d_H_1 = d_H_1_in

    # ReLU 的导数：pre_1 > 0 处为 1，否则为 0
    d_pre_1 = [[v if p > 0.0 else 0.0 for v, p in zip(d_row, p_row)]
               for d_row, p_row in zip(d_H_1, pre_1)]

    # pre_1 = AX · W_0  =>  dW_0 = AXᵀ · d_pre_1
    d_W_0 = matmul_t(AX, d_pre_1)
    for i in range(len(d_W_0)):
        for j in range(len(d_W_0[i])):
            d_W_0[i][j] += lambda_reg * W_0[i][j]

    return d_W_0, d_W_1, L_0

## 5 评估

> `masked_loss` 只统计掩码内节点，是提前停止训练的判据；`accuracy` 用
> `list.index(max(...))` 取 argmax，平局取第一个，与 numpy 的 argmax 一致。

In [6]:
def masked_loss(Z, Y, mask):
    """掩码内的平均交叉熵（不加 L2），早停用的就是它。"""
    total, count = 0.0, sum(mask)
    for i, (z_row, y_row) in enumerate(zip(Z, Y)):
        if mask[i]:
            for f in range(len(z_row)):
                total -= y_row[f] * math.log(z_row[f] + 1e-12)
    return total / count


def accuracy(Z, Y, mask):
    """掩码内的分类准确率。"""
    correct = 0
    for i, (z_row, y_row) in enumerate(zip(Z, Y)):
        if mask[i] and z_row.index(max(z_row)) == y_row.index(max(y_row)):
            correct += 1
    return correct / sum(mask)

## 6 Adam

> 参数只有 C×H + H×F ≈ 2.3 万个，纯 Python 逐元素更新完全够快。

In [7]:
def adam_step(params, t, lr):
    """Adam 更新，逐元素原地做。
    """
    beta_1, beta_2, eps = 0.9, 0.999, 1e-8
    bias_1 = 1.0 - beta_1 ** t
    bias_2 = 1.0 - beta_2 ** t

    for W, d_W, m, v in params:
        for i in range(len(W)):
            W_i, m_i, v_i, g_i = W[i], m[i], v[i], d_W[i]
            for j in range(len(W_i)):
                g = g_i[j]
                m_i[j] = beta_1 * m_i[j] + (1.0 - beta_1) * g
                v_i[j] = beta_2 * v_i[j] + (1.0 - beta_2) * g * g
                m_hat = m_i[j] / bias_1
                v_hat = v_i[j] / bias_2
                W_i[j] -= lr * m_hat / (math.sqrt(v_hat) + eps)

## 7 训练

> 全批量、Adam、早停看**验证损失**（不是准确率），且**不回滚**到最优

In [8]:
def train(AX, A_hat, C, F, Y, train_mask, val_mask, config, rng):
    """全批量训练，Adam 更新，验证损失早停。

    与论文一致：C 是输入通道数，H 是隐藏层特征图数量，F 是类别数。
    """
    H = config["hidden"]

    W_0 = glorot(C, H, rng)             # W^(0) ∈ R^{C×H}
    W_1 = glorot(H, F, rng)             # W^(1) ∈ R^{H×F}

    # Adam 的状态（与参数同形）
    m_W_0, v_W_0 = zeros(C, H), zeros(C, H)
    m_W_1, v_W_1 = zeros(H, F), zeros(H, F)

    keep_prob = 1.0 - config["dropout"]
    best_val_loss = float("inf")
    stale = 0

    for epoch in range(1, config["epochs"] + 1):
        start = time.time()

        # ---- 前向 + 反向 ----
        Z, cache = forward(AX, A_hat, W_0, W_1, keep_prob, rng, training=True)
        d_W_0, d_W_1, L_0 = backward(Z, Y, train_mask, cache,
                                     config["lambda_reg"])

        # ---- Adam ----
        adam_step([(W_0, d_W_0, m_W_0, v_W_0),
                   (W_1, d_W_1, m_W_1, v_W_1)], epoch, config["lr"])

        # ---- 验证（关掉 dropout）----
        Z_val, _ = forward(AX, A_hat, W_0, W_1, 1.0, rng, training=False)
        val_loss = masked_loss(Z_val, Y, val_mask)

        if config["log_every"] and epoch % config["log_every"] == 0:
            print(f"    epoch {epoch:>4d} | L_0 {L_0:.4f} | "
                  f"train_acc {accuracy(Z, Y, train_mask):.4f} | "
                  f"val_loss {val_loss:.4f} | "
                  f"val_acc {accuracy(Z_val, Y, val_mask):.4f} | "
                  f"{time.time() - start:.1f}s/轮", flush=True)

        # ---- 早停：验证损失连续 window 轮不再下降 ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            stale = 0
        else:
            stale += 1
            if stale >= config["early_stopping"]:
                print(f"    early stop at epoch {epoch}，"
                      f"best val_loss {best_val_loss:.4f}", flush=True)
                break

    return W_0, W_1

## 运行

与命令行的调用顺序一致，只是把 `W_0 / W_1 / Z` 留在命名空间里，下一节要用。

> 数据集用 `cora-lite`。

In [9]:
# 只是把训练结果留在 W_0 / W_1 / Z 里，方便下一节继续分析。
config = {"hidden": 16, "epochs": 200, "lr": 0.01, "dropout": 0.5,
          "lambda_reg": 5e-4, "early_stopping": 10, "log_every": 1}

X, A, Y, train_mask, val_mask, test_mask = load_dataset("data/csv/cora-lite")
C, F = len(X[0]), len(Y[0])
print(f"N={len(X)}  C={C}  F={F}  "
      f"train/val/test={sum(train_mask)}/{sum(val_mask)}/{sum(test_mask)}")

X = row_normalize(X)            # 特征按行归一化（论文 §5.1）
A_hat = build_A_hat(A)          # 就地加 I + 对称归一化 D^-1/2 Ã D^-1/2
AX = matmul(A_hat, X)           # 只算一次：X 在训练期不变

rng = random.Random(123)        # 与命令行 --seed 123 一致
W_0, W_1 = train(AX, A_hat, C, F, Y, train_mask, val_mask, config, rng)

Z, _ = forward(AX, A_hat, W_0, W_1, 1.0, rng, training=False)
print(f"final | train_acc {accuracy(Z, Y, train_mask):.4f} | "
      f"val_acc {accuracy(Z, Y, val_mask):.4f} | "
      f"test_acc {accuracy(Z, Y, test_mask):.4f}")

N=250  C=1433  F=7  train/val/test=21/77/152
    epoch    1 | L_0 1.9435 | train_acc 0.1905 | val_loss 1.9427 | val_acc 0.2727 | 0.4s/轮
    epoch    2 | L_0 1.9361 | train_acc 0.2857 | val_loss 1.9408 | val_acc 0.2987 | 0.4s/轮
    epoch    3 | L_0 1.9293 | train_acc 0.3810 | val_loss 1.9385 | val_acc 0.3247 | 0.4s/轮
    epoch    4 | L_0 1.9238 | train_acc 0.3333 | val_loss 1.9357 | val_acc 0.3506 | 0.4s/轮
    epoch    5 | L_0 1.9160 | train_acc 0.4762 | val_loss 1.9325 | val_acc 0.2987 | 0.4s/轮
    epoch    6 | L_0 1.8987 | train_acc 0.4762 | val_loss 1.9288 | val_acc 0.3117 | 0.4s/轮
    epoch    7 | L_0 1.8729 | train_acc 0.6667 | val_loss 1.9248 | val_acc 0.2727 | 0.4s/轮
    epoch    8 | L_0 1.8706 | train_acc 0.6190 | val_loss 1.9207 | val_acc 0.2987 | 0.4s/轮
    epoch    9 | L_0 1.8561 | train_acc 0.6667 | val_loss 1.9164 | val_acc 0.2987 | 0.4s/轮
    epoch   10 | L_0 1.8388 | train_acc 0.6190 | val_loss 1.9120 | val_acc 0.2857 | 0.4s/轮
    epoch   11 | L_0 1.8171 | train_acc 0.571

## 结果分析

总体准确率是一个数，看不出错在哪里。拆到类别层面：哪些类被系统性误判、误判流向哪个类。

In [10]:
# 总体准确率会掩盖结构性问题：拆到每个类别看，再排一张混淆矩阵。
K = len(Y[0])
conf = [[0] * K for _ in range(K)]          # 行 = 真实类别，列 = 预测类别
for i in range(len(Z)):
    if not test_mask[i]:
        continue
    conf[Y[i].index(max(Y[i]))][Z[i].index(max(Z[i]))] += 1

print("类别  测试数  该类准确率")
for k in range(K):
    total = sum(conf[k])
    print(f"{k:>3}   {total:>5}   {conf[k][k] / total:.1%}"
          if total else f"{k:>3}   {total:>5}   -")

print("\n混淆矩阵（行 = 真实，列 = 预测）")
print("      " + "".join(f"{k:>6}" for k in range(K)))
for t in range(K):
    print(f"{t:>4}  " + "".join(f"{v:>6}" for v in conf[t]))

print(f"\n被预测最多的类：{max(range(K), key=lambda k: sum(r[k] for r in conf))}"
      f"（{max(sum(r[k] for r in conf) for k in range(K))} 次）")

类别  测试数  该类准确率
  0      19   15.8%
  1      15   60.0%
  2      22   9.1%
  3      38   86.8%
  4      25   12.0%
  5      17   0.0%
  6      16   18.8%

混淆矩阵（行 = 真实，列 = 预测）
           0     1     2     3     4     5     6
   0       3     3     0    12     1     0     0
   1       1     9     0     5     0     0     0
   2       3     0     2    15     1     1     0
   3       1     3     0    33     1     0     0
   4       1     1     0    20     3     0     0
   5       2     1     0    14     0     0     0
   6       3     1     0     9     0     0     3

被预测最多的类：3（108 次）


## 与命令行对照

| | 本 notebook | `python gcn_simplify.py` |
|---|---|---|
| 数据集 | cora-lite | cora-lite（默认） |
| test_acc | **34.87%** | 34.87% |
| 早停轮数 | 186 | 186 |
| 总耗时 | 约 3 分钟 | 约 3 分钟 |

两边的数字是同一个种子（`--seed 123`）下的确定性结果，重跑逐位一致。

cora-lite 只有 250 个节点、58 条边，它连边都很少，GCN 的邻居聚合几乎没东西可聚合，训练集 21 个节点很快就被记住了（train_acc = 1.0）。
这个数据集的意义是**验证流程跑得通**。